# Python Closures and Decorators — Interview Preparation

Covers: closures, free variables, __closure__, decorator syntax, functools.wraps, stacked decorators, decorators with arguments, class-based decorators, and interview Q&A.

## 1. Closures

In [ ]:
# A closure captures variables from its enclosing scope
def make_counter(start=0):
    count = start  # free variable — captured by the closure

    def counter():
        nonlocal count  # needed to MODIFY (not just read) the free variable
        count += 1
        return count

    return counter  # return the closure


c1 = make_counter()
c2 = make_counter(10)  # independent state

print('c1:', c1(), c1(), c1())  # 1, 2, 3
print('c2:', c2(), c2())        # 11, 12
print('c1 again:', c1())        # 4 — c1 state preserved

In [ ]:
# Inspect closure internals
def make_multiplier(n):
    def multiply(x):
        return x * n  # n is a free variable
    return multiply

double = make_multiplier(2)

print('__closure__:', double.__closure__)
print('captured n:', double.__closure__[0].cell_contents)  # 2
print('free vars:', double.__code__.co_freevars)           # ('n',)

# Each call creates an independent closure
triple = make_multiplier(3)
print('double(5):', double(5))  # 10
print('triple(5):', triple(5))  # 15

## 2. Basic Decorator

In [ ]:
import functools

def my_decorator(func):
    @functools.wraps(func)  # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        print(f'Before {func.__name__}')
        result = func(*args, **kwargs)
        print(f'After {func.__name__}')
        return result
    return wrapper


@my_decorator
def greet(name):
    """Greet someone"""
    print(f'Hello, {name}!')
    return f'Greeted {name}'


# @my_decorator is equivalent to: greet = my_decorator(greet)
result = greet('Alice')
print('Result:', result)
print('__name__:', greet.__name__)  # 'greet' (not 'wrapper') — thanks to @wraps
print('__doc__:', greet.__doc__)

## 3. Practical Decorators

In [ ]:
import time
import functools

# Timer decorator
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'{func.__name__} took {elapsed:.6f}s')
        return result
    return wrapper


# Memoize decorator
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache  # expose cache for inspection
    return wrapper


@timer
@memoize
def fibonacci(n):
    """Fibonacci with memoization"""
    return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)


print('fib(30):', fibonacci(30))
print('fib(30) again:', fibonacci(30))  # instant — cached

## 4. Stacked Decorators

In [ ]:
def decorator_a(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('A: before')
        result = func(*args, **kwargs)
        print('A: after')
        return result
    return wrapper

def decorator_b(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('B: before')
        result = func(*args, **kwargs)
        print('B: after')
        return result
    return wrapper


@decorator_a
@decorator_b
def my_func():
    print('my_func')

# Equivalent to: my_func = decorator_a(decorator_b(my_func))
# Applied bottom-up, executed top-down
print('Calling my_func():')
my_func()

## 5. Decorators with Arguments

In [ ]:
# Three levels of nesting
def repeat(n):
    """Decorator factory — returns a decorator"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(3)
def greet(name):
    print(f'Hello, {name}!')

# Equivalent to: greet = repeat(3)(greet)
greet('Alice')

In [ ]:
# Retry decorator with configurable exceptions
def retry(max_attempts=3, exceptions=(Exception,), delay=0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    print(f'Attempt {attempt} failed: {e}. Retrying...')
                    if delay:
                        time.sleep(delay)
        return wrapper
    return decorator


attempt_count = 0

@retry(max_attempts=3, exceptions=(ValueError,))
def flaky_function():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ValueError(f'Failed on attempt {attempt_count}')
    return 'Success!'


result = flaky_function()
print('Result:', result)

## 6. Class-Based Decorator

In [ ]:
class CountCalls:
    """Class-based decorator that counts function calls"""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.call_count = 0

    def __call__(self, *args, **kwargs):
        self.call_count += 1
        return self.func(*args, **kwargs)

    def reset(self):
        self.call_count = 0


@CountCalls
def add(x, y):
    """Add two numbers"""
    return x + y


add(1, 2)
add(3, 4)
add(5, 6)
print('Call count:', add.call_count)  # 3
print('__name__:', add.__name__)      # 'add'
add.reset()
print('After reset:', add.call_count)  # 0